# SHA3 輸出 NIST SP 800-22：單條百萬 bits 初測資料收集
沿用已驗收的 FPGA 電路，不需 Vivado。這份操作本只收資料；NIST 工具在 Windows 離線執行，不在板子上跑。

請先登記並獨占板子。上一份操作本的 session 必須已安全結束；不要同時執行兩份操作本。

固定收集 3907 批新的硬體摘要，每批 32 bytes，約 5 分鐘（依板況而異）。固定取串接輸出的前 1,000,000 bits 測試，額外 192 bits 也保留。沒有 PRNG 擴增、重複舊資料或選掉不好看的輸出。

這是單條序列初測，不評估多條序列的通過比例／P-value 分布；部分隨機遊走測試可能因回零次數不足顯示 N/A，這不是通過，也不因此重抽。

In [1]:
import json, pynq
from trng_pynq import TrngSession
from trng_sts_collect import validate_addon, collect_pilot
print("PYNQ", pynq.__version__)
assert pynq.__version__.startswith("2.5")
print(json.dumps(validate_addon("."), indent=2))

PYNQ 2.5
{
  "mode": "single_sequence_pilot",
  "target_bits": 1000000,
  "fresh_batches": 3907,
  "build_id": "trng-integrated-20261115T162306Z",
  "entropy_certified": false
}


## 1. 記錄板號與實際日期
維持與原版八檔同一個資料夾。日期填真正操作當天；板上時鐘已知可能不準。

In [3]:
from datetime import datetime
board_id = "21600"
experiment_date = "2026-11-16"
assert board_id.strip()
datetime.strptime(experiment_date, "%Y-%m-%d")
print("Board", board_id, "Experiment", experiment_date)

Board 21600 Experiment 2026-11-16


## 2. 載入同一份已驗收電路
此格會設定 FPGA。不要再開舊操作本載入另一份 overlay。

In [4]:
session = TrngSession("trng_sha3.bit")
print(session.source.snapshot())

Verified release; downloading ONE integrated overlay (RO OFF)
Health source and SHA3 ready at 25 MHz
{'status': 0, 'count': 0, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 0, 'startup_seen': 0}


## 3. 核對 SHA3 並收集 3907 批
先比對 34 個已知訊息，再開始收集。每 128 批顯示一次進度；正常需數分鐘。

每批健康檢查與 FPGA/Python SHA3 必須相符；失敗立即停止並保存證據，沒有自動補抽。這格不管成功或出錯，都會嘗試安全停止。若停止也失敗，保留完整錯誤，不要重複載入電路。

In [5]:
try:
    assert session.self_test(rounds=2) == 34
    pilot = collect_pilot(session, board_id, experiment_date)
    print(json.dumps(pilot, indent=2))
finally:
    session.close()
    tx_status = session.engine._mmio.read(4)
    rx_status = session.engine._mmio.read(0x34)
    print("RO enable:", session.source.mmio.read(32))
    print("DMA halted:", bool(tx_status & 1), bool(rx_status & 1))
    assert session.source.mmio.read(32) == 0 and tx_status & rx_status & 1

PASS SHA3 empty 0
PASS SHA3 abc 3
PASS SHA3 pattern_1 1
PASS SHA3 pattern_3 3
PASS SHA3 pattern_4 4
PASS SHA3 pattern_7 7
PASS SHA3 pattern_31 31
PASS SHA3 pattern_32 32
PASS SHA3 pattern_64 64
PASS SHA3 pattern_135 135
PASS SHA3 pattern_136 136
PASS SHA3 pattern_137 137
PASS SHA3 pattern_271 271
PASS SHA3 pattern_272 272
PASS SHA3 pattern_273 273
PASS SHA3 pattern_1000 1000
PASS SHA3 pattern_4096 4096
PASS SHA3 empty 0
PASS SHA3 abc 3
PASS SHA3 pattern_1 1
PASS SHA3 pattern_3 3
PASS SHA3 pattern_4 4
PASS SHA3 pattern_7 7
PASS SHA3 pattern_31 31
PASS SHA3 pattern_32 32
PASS SHA3 pattern_64 64
PASS SHA3 pattern_135 135
PASS SHA3 pattern_136 136
PASS SHA3 pattern_137 137
PASS SHA3 pattern_271 271
PASS SHA3 pattern_272 272
PASS SHA3 pattern_273 273
PASS SHA3 pattern_1000 1000
PASS SHA3 pattern_4096 4096
SUCCESS 34 known-message FPGA SHA3 comparisons
PASS fresh hardware digests 128/3907; RO OFF
PASS fresh hardware digests 256/3907; RO OFF
PASS fresh hardware digests 384/3907; RO OFF
PASS f

## 4. 下載
成功摘要應顯示 PASS_COLLECTION、completed_batches=3907、generated_bits=1000192、unused_tail_bits=192。

Ctrl+S 儲存。回 Jupyter 檔案頁，進入 sts_pilot_results，一次勾選本次 ZIP → Download；另以 File → Download as → Notebook (.ipynb) 下載這份含輸出的操作本。

將 ZIP 和操作本交回，才能執行官方 STS 的 15 類測試。請連同失敗結果保存，不要重抽直到全 PASS。